<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

In [ ]:
# Common Declarations and setup

import os
import sys
sys.path.insert(0, '../src')
import time
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from io import BytesIO

%matplotlib widget

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi import tddn
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output 

from dataclasses import dataclass, fields
from typing import List



# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

display(Markdown(f"Phaser root: **{phaser_root}**"))
display(Markdown(f"Resource path: **{resource_path}**\n"))

In [ ]:
@dataclass
class RadarConfig:
    """Configuration parameters for FMCW radar operation."""

    # ========== SDR Parameters ==========
    sample_rate: float = 5.85e6  # NOTE: This is a placeholder! The actual sample rate is calculated
                                  # in configure_sdr() based on buffer_size / frame_time to ensure
                                  # we capture exactly one complete frame (chirp + padding).
                                  # Formula: sample_rate = sdr_buf_size / (ramp_time + pri_padding_ms)
                                  # For 4096 samples / 0.7ms = 5.85 MHz
                                  # ALWAYS use sdr.sample_rate (actual) not config.sample_rate (default) in plots!

    center_freq: float = 2.1e9  # SDR LO frequency (Hz). Upconverted to output_freq by ADF4159
                                 # Keep at 2.1 GHz for optimal Pluto performance

    signal_freq: float = 100e3  # TX baseband tone frequency (Hz). Creates IF offset
                                 # Used to separate DC offset from target returns
                                 # Typical: 100 kHz. Don't change unless you know why

    rx_gain: int = 30  # Receiver gain (dB). Range: -3 to 70 dB
                        # Higher = more sensitive but risk ADC saturation on strong returns
                        # Start at 60, reduce if seeing saturation artifacts
                        # Trade-off: +10 dB gain ~= 3x detection range OR 10 dB less TX power needed

    tx_gain: int = 0  # Transmitter gain (dB). Range: 0 to -88 dB (0 = max power)
                       # 0 dB ~= +30 dBm EIRP with array gain ~= 1W effective
                       # Reduce for short range, regulations, or power saving
                       # Trade-off: -6 dB power ~= 0.5x detection range

    sdr_buf_size: int = 1024 * 8
    fft_size: int = sdr_buf_size

    # ========== Chirp Parameters ==========
    output_freq: float = 9.9e9  # Radar transmit frequency (Hz). X-band (8-12 GHz)
                                 # 9.9 GHz = 30mm wavelength. Good for small targets
                                 # Check local regulations (ISM, amateur, Part 15)

    chirp_BW: float = 500e6  # Chirp bandwidth (Hz). Determines range resolution
                              # Range resolution = c/(2*BW) = 0.3m at 500 MHz
                              # Wider BW = better resolution, more processing
                              # Typical: 250 MHz to 1 GHz (if hardware supports)
                              # Trade-off: 2x BW = 0.5x range resolution (better)

    ramp_time_us: int = 500  # Chirp duration (microseconds). Affects max range
                           # Longer = more samples per chirp = better range resolution
                           # Also affects PRF (pulse repetition frequency)
                           # Typical: 100 to 1000 us
                           # Trade-off: 2x ramp_time_us = 0.5x PRF = 0.5x max unambiguous velocity

    num_chirps: int = 2  # Number of chirps per frame (CPI - Coherent Processing Interval)
                            # More chirps = better Doppler (velocity) resolution
                            # Doppler resolution = lambda/(2*CPI*PRI) where PRI ~= ramp_time_us
                            # Typical: 64 to 512. Power of 2 for efficient FFT
                            # Trade-off: 2x chirps = 0.5x Doppler resolution, 2x processing time

    # ========== Array Parameters ==========
    element_spacing: float = 0.014  # Antenna element spacing (meters). 14mm ~= lambda/2 at 10 GHz
                                     # lambda/2 spacing prevents grating lobes (spatial aliasing)
                                     # Don't change unless physical array changes

    gain_list: List[int] = None  # Per-element gain (0-127). None = all max (127)
                                  # Can apply taper (Blackman, Taylor) to reduce sidelobes
                                  # Example: [8, 34, 84, 127, 127, 84, 34, 8] for Blackman
                                  # Trade-off: Tapering reduces sidelobes but lowers gain

    # ========== Timing Parameters (Advanced) ==========
    begin_offset_fraction: float = 0.1  # Fraction of chirp to skip at start (0.0-0.3)
                                         # VCO takes time to settle; early samples are non-linear
                                         # 0.1 = skip first 10% of chirp (30 us at 300 us ramp)
                                         # Increase if seeing range artifacts near zero
                                         # Trade-off: More offset = fewer samples = less SNR

    pri_padding_ms: float = 0.1   # Dead time between chirps (milliseconds)
                                  # Allows VCO to reset and prevents chirp overlap
                                  # PRI (Pulse Repetition Interval) = ramp_time_us + padding
                                  # Affects PRF and max unambiguous velocity
                                  # Trade-off: More padding = lower PRF = lower max velocity

    # ========== TDD (Time Division Duplex) Parameters ==========
    tdd_trigger_on_raw: int = 0   # TDD GPIO trigger start (raw units)
                                   # Synchronizes chirp generation with data capture
                                   # Keep at 0 for immediate trigger

    tdd_trigger_off_raw: int = 20  # TDD GPIO trigger stop (raw units)
                                    # Pulse width for trigger signal
                                    # Typical: 5-20. Must be long enough for hardware to latch
                                    # Trade-off: Longer pulse more reliable but delays start

    rpi_ip:       str = "192.168.1.10"
    #rpi_ip:       str = "phaser.local"
    sdr_ip:       str = "192.168.2.1"
    fieldfox_ip:  str = "192.168.1.30"

    def __post_init__(self):
        """Set default gain list and calibration file paths if not provided."""
        if self.gain_list is None:
            self.gain_list = [127] * 8

    def __iter__(self):
        for field in fields(self):
            yield field.name, getattr(self, field.name)

#  Default are stored in a dataclass
config = RadarConfig()

display(Markdown("#### Config values"))
for name, value in config:    
    display(Markdown(f"{name} = {value}"))

pll    = None      
gpio   = None
tdd    = None
phaser = None
sdr    = None


In [ ]:
def connect_devices():
    md = """ """
    try:
        global pll, gpio, tdd, phaser, sdr
        display(Markdown(f"- ADF4159: ip: {config.rpi_ip}"))
        pll    = adf4159        (uri="ip:" + config.rpi_ip)
        
        display(Markdown(f"- GPIO: ip: {config.sdr_ip}"))
        gpio   = one_bit_adc_dac(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- TDDN: ip: {config.sdr_ip}"))
        tdd = tddn(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- Phaser: ip: {config.rpi_ip}"))
        phaser = CN0566         (uri="ip:" + config.rpi_ip)

        display(Markdown(f"- SDR: ip: {config.sdr_ip}"))
        sdr    = ad9361         (uri="ip:" + config.sdr_ip)
    
    except:
        display(Markdown(f"Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
        print("")
        sys.exit(1)

    phaser.sdr = sdr
    
def configure_phaser():
    phaser.configure(device_mode="rx")
    phaser.element_spacing = config.element_spacing
    
    for i in range(0, 8):
        phaser.set_chan_phase(i, 0)

    display(Markdown(f"- Phaser channel phase  = 0"))
    
    for i in range(0, len(config.gain_list)):
        phaser.set_chan_gain(i, config.gain_list[i], apply_cal=False)
    
    phaser._gpios.gpio_tx_sw = 0
    phaser._gpios.gpio_vctrl_1 = 1
    phaser._gpios.gpio_vctrl_2 = 1

def configure_sdr():   
    destroy_sdr_buffer()
    
    # Configure sample rate to capture the full frame (chirp + padding) x no of chirps
    # 
    frame_time_s = ( (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3) ) * config.num_chirps

    sr = int(config.sdr_buf_size / frame_time_s)
    phaser.sdr.sample_rate = sr
    
    phaser.sdr.rx_lo = int(config.center_freq)
    phaser.sdr.rx_enabled_channels = [0, 1]
    phaser.sdr.rx_buffer_size = config.sdr_buf_size
    
    phaser.sdr.gain_control_mode_chan0 = 'manual'
    phaser.sdr.gain_control_mode_chan1 = 'manual'
    phaser.sdr.rx_hardwaregain_chan0 = -88
    phaser.sdr.rx_hardwaregain_chan1 = config.rx_gain

    phaser.sdr.tx_buffer_size = config.sdr_buf_size
    phaser.sdr.tx_lo = int(config.center_freq)
    phaser.sdr.tx_enabled_channels = [0, 1]
    phaser.sdr.tx_cyclic_buffer = True
    phaser.sdr.tx_hardwaregain_chan0 = -88
    phaser.sdr.tx_hardwaregain_chan1 = int(config.tx_gain)

    display(Markdown(f"""- Setting sample  rate: {sr/1e6:.2f} MHz
- Actual Sample rate: {sdr.sample_rate/1e6:.2f} MHz
- Frame time: {frame_time_s*1e3:.2f} ms (chirp + padding) * Num Chirps
- Chirp time: {config.ramp_time_us} us
- Padding time: {config.pri_padding_ms} ms
- RX LO: {sdr.rx_lo/1e9:.1f} GHz
- TX LO: {sdr.tx_lo/1e9:.1f} GHz
- Buffer size: {sdr.rx_buffer_size} samples
- Capture time: {sdr.rx_buffer_size/sdr.sample_rate*1e3:.2f} ms"""))

def destroy_sdr_buffer():
    try: sdr.tx_destroy_buffer()
    except: pass
    try: sdr.rx_destroy_buffer()
    except: pass

def configure_adf4159():
    # Configure ADF4159 for triggered sawtooth chirp

    vco_freq = int(config.output_freq + config.signal_freq + config.center_freq)
    BW = config.chirp_BW
    num_steps = int(config.ramp_time_us)

    phaser.frequency = int(vco_freq / 4)
    phaser.freq_dev_range = int(BW / 4)
    phaser.freq_dev_step = int((BW / 4) / num_steps)
    phaser.freq_dev_time = int(config.ramp_time_us)
    
    phaser.delay_word = 4095
    phaser.delay_clk = "PFD"
    phaser.delay_start_en = 0
    phaser.ramp_delay_en = 0
    phaser.trig_delay_en = 0
    phaser.ramp_mode = "single_sawtooth_burst"
    phaser.sing_ful_tri = 0
    phaser.tx_trig_en = 1
    phaser.enable = 0                                  # 0 = PLL enable.  Write this last to update all the registers

    display(Markdown(f"""- VCO Output Frequency = {vco_freq/1e9} GHz
- Chirp BW: {(4 * phaser.freq_dev_range)/1e6:.0f} MHz
- Ramp time: {config.ramp_time_us} us
- Chirp rate: {config.chirp_BW/(config.ramp_time_us*1e-6)/1e12:.2f} THz/s"""))

def configure_tdd():
    """
    Configure the TDD (Time Division Duplex) controller so that each FMCW chirp
    is synchronised to the data acquisition hardware.

    The TDD engine generates trigger signals at the start of each chirp period
    (PRI) and repeats this for the required number of chirps in a burst.
    """

    # Route the TDD trigger to the external sync circuitry and enable
    # the Phaser board trigger path.
    gpio.gpio_tdd_ext_sync = True
    gpio.gpio_phaser_enable = True

    # Disable the TDD engine while its configuration is updated.
    tdd.enable = False

    # Use an external trigger source to start the TDD sequence.
    tdd.sync_external = True

    # Begin generating triggers immediately after synchronisation.
    tdd.startup_delay_ms = 0

    # Calculate the Pulse Repetition Interval (PRI).
    #
    # The PRI consists of:
    #   - the FMCW ramp (chirp) duration
    #   - additional padding time between chirps
    #
    # ramp_time_us is stored in microseconds, so convert to milliseconds
    # before adding the padding value.
    PRI_ms = (config.ramp_time_us / 1e3) + config.pri_padding_ms

    
    # Set the time between successive chirp triggers.
    tdd.frame_length_ms = PRI_ms
    
    # Generate one trigger event per chirp in the burst.
    tdd.burst_count = config.num_chirps

    tdd.channel[0].enable = True
    tdd.channel[0].polarity = False
    tdd.channel[0].on_raw = config.tdd_trigger_on_raw
    tdd.channel[0].off_raw = config.tdd_trigger_off_raw

    tdd.channel[1].enable = True
    tdd.channel[1].polarity = False
    tdd.channel[1].on_raw = config.tdd_trigger_on_raw
    tdd.channel[1].off_raw = config.tdd_trigger_off_raw
    
    tdd.channel[2].enable = True
    tdd.channel[2].polarity = False
    tdd.channel[2].on_raw = config.tdd_trigger_on_raw
    tdd.channel[2].off_raw = config.tdd_trigger_off_raw
    
    # Apply the configuration and start the TDD engine.
    tdd.enable = True

    display(Markdown(f"""- Frame Len =  {tdd.frame_length_ms} ms
- No. Chirps / trigger =  {config.num_chirps}"""))

def tx_baseband():
    # Generate baseband transmit waveform (tone at IF)

    fs = int(sdr.sample_rate)
    N = config.sdr_buf_size
    t = np.arange(N) / fs  # FIXED: Guarantees exactly N samples
    
    i_data = np.cos(2 * np.pi * t * config.signal_freq) * 2**14
    q_data = np.sin(2 * np.pi * t * config.signal_freq) * 2**14
    iq_data = i_data + 1j * q_data

    # Validate buffer length BEFORE sending to SDR
    expected_tx_buffer = sdr.tx_buffer_size
    actual_samples = len(iq_data)
    
    display(Markdown(f"- Transmit {config.signal_freq/1e3} kHz"))
    display(Markdown(f"- TX buffer size: {expected_tx_buffer} (expected) vs {actual_samples} (actual)"))
    
    if actual_samples != expected_tx_buffer:
        raise ValueError(f"❌ Buffer length mismatch! Expected {expected_tx_buffer}, got {actual_samples}")
    
    # Send waveform to TX buffer
    sdr.tx([iq_data, iq_data])
    
    display(Markdown(f"- TX buffer loaded successfully"))

# FMCW RADAR: Range-Doppler Processing (2D FFT)

## Overview

In the previous notebook, we successfully measured the **range** of static targets using synchronized FMCW chirps and the Range FFT.
Range is useful, but so is **velocity** and **bearing**.

In this notebook we will focus on measuring velocity.

## Learning Objectives

By the end of this notebook, you will:
1. Understand the concept of **fast-time** vs **slow-time** in FMCW RADAR
2. Capture a **Coherent Processing Interval (CPI)** of multiple chirps
3. Implement the **2D FFT** for Range-Doppler processing
4. Generate and interpret **Range-Doppler Maps (RDM)**
5. Extract both range and velocity from moving targets
6. Understand range-velocity ambiguities and coupling
7. Apply MTI (Moving Target Indication) filtering

# 1: Velocity in Radar Systems

In radar systems, velocity refers to radial velocity: the component of a target's motion along the radar's line of sight. A target moving directly towards or away from the radar produces a measurable Doppler shift, while a target moving perpendicular to the radar may exhibit little or no Doppler shift. Consequently, Doppler processing measures how quickly a target's range is changing, rather than its complete three-dimensional velocity vector.  

$
\large
v_r = \frac{dR}{dt}
$

where:

R = target range  
$v_r$ = radial velocity

## Why is this important?

When we move from Range FFT to Doppler processing, we're not measuring:

    "How fast is the target moving through space?"

We're measuring:

    "How fast is the target moving towards or away from the radar?"

The Doppler FFT estimates this radial velocity by observing the phase progression of the target return across multiple chirps.

## Can we use the FFT Range Data to Measure Velocity

If velocity is simply the rate of change of range, couldn't we estimate it by tracking how the range peak moves between chirps?
The answer is yes, but with some important limitations.

Let's use the FMCW configuration from the previous notebook:

- Chirp length = 0.5mS
- Padding = 0.1mS

To measure velocity, we need to compare multiple chirps. The **Pulse Repetition Time** (PRT) used in the previous notebook was 0.6mS (ramp time + padding)

Now consider a target travelling at 30 m/s.
The distance travelled between two consecutive chirps is:

$
\large
\begin{aligned} \\
\Delta R &= vT_c \\ 
&= 30 \times 0.6\text{ ms} \\ 
&= 18\text{ mm} 
\end{aligned}
$

Recall that our range resolution was approximately:

$
\large
\begin{aligned} \\
\Delta R &= frac{c}{2 \times BW} \\
         &= \frac{3e^8}{2 \times 500e^6}
         &= 0.3m
\end{aligned}
$  

Comparing the two values:

Target movement per chirp = 18 mm
Range resolution = 300 mm

The target moves only about 6% of a range bin between chirps.

As a result, the range peak appears almost stationary in the FFT magnitude plot, making velocity estimation from peak movement both difficult and noisy.

### Could We Improve This?

A few intuitive options come to mind:

#### Increast PRT

This would give the target more time to move between chirps, making changes in range easier to observe.
However, this comes at a cost:

- Slower update rate
- Longer frame times
- Reduced responsiveness to changing targets

#### Increase the chirp Bandwidth

Increasing bandwidth improves range resolution:

However:
- Large bandwidths may not be available
- Higher bandwidth often increases hardware cost and complexity
- We'd still be relying solely on changes in FFT magnitude

#### There's a Better Way 

So far, we've only considered the magnitude of the Range FFT.  
However, every FFT bin also contains phase information.  
While the magnitude tells us how much energy exists in a particular range bin, the phase tells us something equally valuable:

    Exactly where the target lies within that range bin.  

A target may move only a few millimetres, producing almost no visible change in FFT magnitude, yet that same movement can produce a measurable phase change between consecutive chirps.

By analysing how this phase evolves over time, we can estimate velocity far more accurately than by tracking movement of the range peak alone.

This insight forms the foundation of Doppler processing.

# 2: Doppler Frequency and Velocity Theory

## Doppler Effect Refresher

When a target moves relative to the RADAR, the received frequency shifts:

$
\Large
f_d = \frac{2 v \cdot f_c}{c}
$

Where:
- $f_d$ = Doppler frequency shift (Hz)
- $v$ = target radial velocity (m/s) (positive = approaching)
- $f_c$ = carrier frequency (Hz)
- $c$ = speed of light (m/s)


## FMCW Doppler: Phase Change Across Chirps

After the Range FFT, a target appears in a specific range bin. If the target is moving, the complex value in that bin exhibits a phase progression from chirp to chirp.

The phase change between successive chirps is:

$
\Large
\Delta \phi = 2\pi f_d T_{\text{chirp}}
$

where:

- $f_d$ is the Doppler frequency
- $T_{\text{PRI}}$ is the time between chirp starts (Pulse Repetition Interval)

A stationary target has approximately constant phase, while a moving target produces a linear phase ramp across chirps.
The Doppler FFT detects this phase progression and converts it into a velocity estimate.

## Demo 

Lets run the previous demo, were we looked a the FFT range plot.
We will remove the spectrogram and add a plot showing the phaser difference two consecutive plots

We're going to use the TDD engine to help with timing again. You may have noticed:

`tdd.burst_count = 1`I

Setting this property gives us the number of chirps that will be generated by TDD system it will ensure all timings are adhered too.

Lets set `tdd.burst_count = 2`.

We will calculated the avergare of the magnitude and the difference in the phases.


In [ ]:
# ============================================================================
# Hardware Configuration
# ============================================================================

display(Markdown("**Connect to Devices**"))
connect_devices()

display(Markdown("**Configure Phaser**"))
configure_phaser()

display(Markdown("**Configure PlutoSDR**"))
destroy_sdr_buffer()
configure_sdr()

display(Markdown("**Configure ADF4159**"))
configure_adf4159()

display(Markdown("**Configure TDD Engine**"))
configure_tdd()

display(Markdown("**Transmit Baseband Signal**"))
tx_baseband()


In [ ]:
def capture_data():
    """Trigger TDD and capture synchronized data."""
    phaser._gpios.gpio_burst = 0
    phaser._gpios.gpio_burst = 1
    phaser._gpios.gpio_burst = 0
    time.sleep(0.001)
    return sdr.rx()

def process_data(raw_data):
    """Combine both RX channels."""
    return raw_data[0] + raw_data[1]

def extract_chirps(rx_data, num_chirps):
    """Extract individual chirps from received buffer."""
    actual_sample_rate = sdr.sample_rate

    # Calculate expected samples per chirp period (PRI = ramp + padding)
    PRI_time_s = (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3)
    samples_per_PRI = int(actual_sample_rate * PRI_time_s)
    total_samples_needed = samples_per_PRI * num_chirps
    
    # Calculate samples in just the ramp (no padding)
    chirp_samples = int(actual_sample_rate * config.ramp_time_us * 1e-6)
    
    # Validation check
    buffer_size = len(rx_data)
    if buffer_size < total_samples_needed:
        raise ValueError(
            f"Buffer too small for {num_chirps} chirps!"
            f"  Buffer size: {buffer_size} samples"
            f"  Needed: {total_samples_needed} samples"
            f"  Per chirp (PRI): {samples_per_PRI} samples ({PRI_time_s*1e3:.2f} ms)"
            f"  Sample rate: {actual_sample_rate/1e6:.2f} MHz"
            f"  Solution: Increase config.sdr_buf_size to at least {total_samples_needed}"
        )
    
    # Divide buffer into num_chirps sections
    samples_per_section = buffer_size // num_chirps
    
    # Make sure chirp_samples doesn't exceed section size
    chirp_samples = min(chirp_samples, samples_per_section)
    
    # Extract the first chirp_samples from each section
    chirps = []
    for i in range(num_chirps):
        start = i * samples_per_section
        end = start + chirp_samples
        chirp = rx_data[start:end]
        chirps.append(chirp)
    
    return np.array(chirps)

def process_chirps(chirps_2d):
    """Return FFT magnitude and phase for each chirp."""
    magnitudes = []
    phases = []
    for chirp in chirps_2d:
        window = np.blackman(len(chirp))
        spectrum = np.fft.fft(chirp * window)
        spectrum = spectrum[: len(spectrum) // 2]
        magnitudes.append(np.abs(spectrum))
        phases.append(np.angle(spectrum))
    return np.array(magnitudes), np.array(phases)


def render_plot():
    """Render the current figure as PNG bytes for the image widget."""
    with BytesIO() as buffer:
        canvas.print_png(buffer)
        return buffer.getvalue()

# ============================================================================
# Create Output Widgets for Layout
# ============================================================================

output_status = widgets.Output(layout=widgets.Layout(
    width='100%',
    border='1px solid #ddd',
    padding='10px'
))

output_info = widgets.Output(layout=widgets.Layout(
    width='100%',
    height='180px',
    border='1px solid #ddd',
    padding='10px',
    overflow='auto'
))

# ============================================================================
# Initial Information (captured in output widget)
# ============================================================================

with output_status:
    print("Starting FMCW Doppler Demo")
    print("Press Ctrl+C (Interrupt kernel) to stop\n")
    print(f"Sample Rate : {sdr.sample_rate/1e6:.2f} MHz")
    print(f"Buffer Size : {config.sdr_buf_size}")
    print(f"Chirps      : {config.num_chirps}")
    print("\nWarming up hardware...")

try:
    _ = capture_data()
except:
    pass

with output_status:
    print("Ready\n")

# ============================================================================
# Plot Setup
# ============================================================================


plt.close('all')

# Render PNG frames independently of the notebook's interactive backend.
fig = Figure(figsize=(14, 14), dpi=90)
canvas = FigureCanvasAgg(fig)
ax_raw, ax1, ax2 = fig.subplots(3, 1)

# Expected timing only: sample zero is assumed to align with the first trigger.
# Use the TDD period readback; the ramp duration is the configured value.
raw_sample_rate = float(sdr.sample_rate)
expected_pri_us = float(tdd.frame_length_ms) * 1e3
expected_ramp_us = float(config.ramp_time_us)
settling_us = config.begin_offset_fraction * expected_ramp_us
line_raw_i, = ax_raw.plot([], [], color='tab:blue', linewidth=0.6, alpha=0.7, label='I (RX0 + RX1)')
line_raw_q, = ax_raw.plot([], [], color='tab:orange', linewidth=0.6, alpha=0.7, label='Q (RX0 + RX1)')
line_raw_abs, = ax_raw.plot([], [], color='black', linewidth=0.9, label='Magnitude')
for chirp_index in range(config.num_chirps):
    start_us = chirp_index * expected_pri_us
    end_us = start_us + expected_ramp_us
    ax_raw.axvline(start_us, color='tab:green', linestyle='--', linewidth=1.2,
                   label='Expected ramp start' if chirp_index == 0 else None)
    ax_raw.axvline(end_us, color='tab:red', linestyle='--', linewidth=1.2,
                   label='Expected ramp end' if chirp_index == 0 else None)
    ax_raw.axvspan(start_us, start_us + settling_us, color='gold', alpha=0.2,
                   label='Configured settling interval' if chirp_index == 0 else None)
    ax_raw.axvspan(end_us, start_us + expected_pri_us, color='grey', alpha=0.15,
                   label='Expected padding' if chirp_index == 0 else None)
    ax_raw.text(start_us + expected_ramp_us / 2, 0.97, f'Chirp {chirp_index + 1}',
                transform=ax_raw.get_xaxis_transform(), ha='center', va='top', fontsize=9)
ax_raw.set_xlabel('Time from first RX sample (us)', fontsize=11)
ax_raw.set_ylabel('Raw amplitude (ADC counts)', fontsize=11)
ax_raw.set_title('Raw dechirped I/Q - assuming first trigger at t = 0', fontsize=13, fontweight='bold')
ax_raw.set_xlim(0, sdr.rx_buffer_size / raw_sample_rate * 1e6)
ax_raw.grid(True, alpha=0.3)
ax_raw.legend(loc='lower left', fontsize=8, ncol=4)


line_mag, = ax1.plot([60, 80, 100, 120, 140], [-80, -60, -40, -60, -80], 'b-', linewidth=1)
peak_marker1, = ax1.plot([100], [-40], 'ro', markersize=10)
line_phase, = ax2.plot([60, 80, 100, 120, 140], [-100, -50, 0, 50, 100], 'r-', linewidth=1.5)
peak_marker2, = ax2.plot([100], [0], 'ro', markersize=10)

peak_range_label = ax1.annotate(
    '', xy=(100, -40), xytext=(12, -12), textcoords='offset points',
    fontsize=10, fontweight='bold', color='darkred', va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9),
    annotation_clip=True
)
peak_velocity_label = ax2.annotate(
    '', xy=(100, 0), xytext=(12, 12), textcoords='offset points',
    fontsize=10, fontweight='bold', color='darkred', va='bottom',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red', alpha=0.9),
    annotation_clip=True
)

ax1.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax1.set_ylabel('Magnitude (dB)', fontsize=11)
ax1.set_title('FFT Spectrum', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([100, 150])
ax1.set_ylim([-100, 0])

ax2.set_xlabel('Beat Frequency (kHz)', fontsize=11)
ax2.set_ylabel('Phase Difference (degrees)', fontsize=11)
ax2.set_title('Phase Difference Between Chirps', fontsize=13, fontweight='bold')
ax2.set_ylim([-10, 10])
ax2.set_xlim([100, 150])
ax2.grid(True, alpha=0.3)

fig.tight_layout()


# Keep PNG rendering and place the image alongside the other output widgets.
plot_widget = widgets.Image(
    value=render_plot(),
    format='png',
    layout=widgets.Layout(width='100%', max_width='1260px', height='auto')
)

demo_ui = widgets.VBox([
    widgets.HTML("<h3 style='margin:10px 0;'>FMCW Doppler Processing</h3>"),
    output_status,
    plot_widget,
    widgets.HTML("<h4 style='margin:10px 0;'>Target Info</h4>"),
    output_info,
], layout=widgets.Layout(width='100%'))
display(demo_ui)

# ============================================================================
# Phase Calibration
# ============================================================================
PHASE_CALIBRATION_DEG = 60

# ============================================================================
# Main Loop
# ============================================================================

frame_count = 0

try:
    while True:
        frame_count += 1

        raw_data = capture_data()
        rx_data = process_data(raw_data)

        # Show the complete, unwindowed buffer before chirp extraction.
        raw_time_us = np.arange(len(rx_data)) / raw_sample_rate * 1e6
        line_raw_i.set_data(raw_time_us, rx_data.real)
        line_raw_q.set_data(raw_time_us, rx_data.imag)
        line_raw_abs.set_data(raw_time_us, np.abs(rx_data))
        raw_limit = max(float(np.max(np.abs(rx_data))), 1.0)
        ax_raw.set_ylim(-1.1 * raw_limit, 1.1 * raw_limit)
        ax_raw.set_xlim(0, len(rx_data) / raw_sample_rate * 1e6)
        ax_raw.set_title(
            f'Raw dechirped I/Q - Frame {frame_count} (assuming first trigger at t = 0)',
            fontsize=13, fontweight='bold'
        )
        chirps = extract_chirps(rx_data, num_chirps=config.num_chirps)
        magnitudes, phases = process_chirps(chirps)

        actual_sample_rate = sdr.sample_rate
        avg_magnitude = np.mean(magnitudes, axis=0)
        magnitude_db = 20 * np.log10(avg_magnitude + 1e-12)

        fft_size = len(avg_magnitude) * 2
        freqs_hz = np.fft.fftfreq(fft_size, 1/actual_sample_rate)[:len(avg_magnitude)]
        freqs_khz = freqs_hz / 1e3

        chirp_time_s = config.ramp_time_us * 1e-6
        chirp_rate = config.chirp_BW / chirp_time_s
        IF_freq = config.signal_freq
        beat_freq = np.abs(freqs_hz - IF_freq)
        range_bins_m = (3e8 * beat_freq) / (2 * chirp_rate)

        target_bin = np.argmax(magnitude_db)

        phase_diff_rad = np.diff(phases, axis=0).mean(axis=0)
        phase_diff_rad = np.arctan2(np.sin(phase_diff_rad), np.cos(phase_diff_rad))
        phase_diff_deg = np.degrees(phase_diff_rad)
        
        phase_diff_deg = phase_diff_deg + PHASE_CALIBRATION_DEG
        phase_diff_deg = np.arctan2(np.sin(np.radians(phase_diff_deg)), 
                                      np.cos(np.radians(phase_diff_deg)))
        phase_diff_deg = np.degrees(phase_diff_deg)
        
        target_phase_diff_rad = np.radians(phase_diff_deg[target_bin])
        target_phase_diff_deg = phase_diff_deg[target_bin]

        peak_freq = freqs_khz[target_bin]

        TPRI = (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3)
        wavelength = (3e8 / config.output_freq)
        fd = target_phase_diff_rad / (2 * np.pi * TPRI)
        velocity = wavelength * fd / 2

        line_mag.set_data(freqs_khz, magnitude_db)
        ax1.set_ylim([np.max(magnitude_db)-60, np.max(magnitude_db)+5])
        ax1.set_title(f'FFT Spectrum - Frame {frame_count}', fontsize=13, fontweight='bold')

        peak_mag = magnitude_db[target_bin]
        peak_marker1.set_data([peak_freq], [peak_mag])

        line_phase.set_data(freqs_khz, phase_diff_deg)
        ax2.set_title(f'Phase Difference - Frame {frame_count}', fontsize=13, fontweight='bold')

        peak_phase = phase_diff_deg[target_bin]
        peak_marker2.set_data([peak_freq], [peak_phase])

        # Keep the labels beside their markers and directed into the plot.
        for axis, label, peak_y, label_text in (
            (ax1, peak_range_label, peak_mag, f'{range_bins_m[target_bin]:.2f} m'),
            (ax2, peak_velocity_label, peak_phase, f'{velocity:+.2f} m/s'),
        ):
            label.xy = (peak_freq, peak_y)
            label.set_text(label_text)
            x_low, x_high = axis.get_xlim()
            y_low, y_high = axis.get_ylim()
            right_half = peak_freq > (x_low + x_high) / 2
            upper_half = peak_y > (y_low + y_high) / 2
            label.set_position((-12 if right_half else 12, -12 if upper_half else 12))
            label.set_ha('right' if right_half else 'left')
            label.set_va('top' if upper_half else 'bottom')

        plot_widget.value = render_plot()

        # Yield between captures so the notebook can process display updates.
        await asyncio.sleep(0.1)

        with output_info:
            clear_output(wait=True)
            print(f"Frame: {frame_count:5d}  |  Peak: {peak_freq:6.2f} kHz (bin {target_bin})")
            print(f"{'='*70}")
            print(f"  Target Range:    {range_bins_m[target_bin]:8.2f} m")
            print(f"  Phase Diff:      {target_phase_diff_deg:8.1f} °")
            print(f"  Doppler Shift:   {fd:8.1f} Hz")
            print(f"  Velocity:        {velocity:8.2f} m/s")
            print(f"{'='*70}")

except KeyboardInterrupt:
    with output_info:
        clear_output(wait=True)
        print("\n" + "="*70)
        print("  INTERRUPTED")
        print("="*70)

finally:
    plt.close(fig)
    with output_status:
        print(f"\nCapture stopped - Total frames: {frame_count}")


## Velocity Resolution

$
\Large
\Delta v = \frac{\lambda}{2 \cdot M \cdot T_{\text{chirp}}}
$

Where:
- $\lambda$ = wavelength
- $M$ = number of chirps in CPI

**More chirps → better velocity resolution** (but longer CPI duration)

## Maximum Unambiguous Velocity

$
\Large
v_{\text{max}} = \frac{\lambda}{4 \cdot T_{\text{chirp}}} = \frac{\lambda \cdot PRF}{4}
$

Velocities beyond $v_{\text{max}}$ alias (appear at wrong velocity)

1. **No velocity information**: We couldn't determine if a target was moving or how fast
2. **Single chirp processing**: We processed each chirp independently with no coherent integration

This notebook introduces **Range-Doppler processing** using the **2D FFT**, which simultaneously measures both **range** and **velocity** of multiple targets.

<div style="text-align:center;">
  <img src="resources/range_doppler.svg" alt="Phaser Block Diagram" height="300">
</div>

## Section 1: Fast-Time vs Slow-Time

### The Two Time Dimensions in FMCW

FMCW RADAR operates in **two time domains**:

#### Fast-Time (Within a Chirp)
- Time **within** a single chirp
- Sample rate: $f_s$ (e.g., 600 kHz)
- Used to measure **range** via beat frequency
- FFT along fast-time → Range FFT

#### Slow-Time (Across Chirps)
- Time **between** chirps
- Sample rate: Pulse Repetition Frequency (PRF)
- Used to measure **velocity** via Doppler frequency
- FFT along slow-time → Doppler FFT

### The 2D Data Matrix

When we capture $M$ chirps of $N$ samples each, we build a 2D matrix:

<div style="text-align:center;">
  <img src="resources/range_doppler_columns.svg" alt="Phaser Block Diagram" height="300">
</div>

- **Rows (slow-time)**: Different chirps
- **Columns (fast-time)**: Samples within each chirp

### The 2D FFT Process

1. **Range FFT** (along columns): $N$-point FFT for each chirp → Range profiles
2. **Doppler FFT** (along rows): $M$-point FFT for each range bin → Doppler information

Result: **Range-Doppler Map** (RDM) showing targets in range-velocity space

## Section 3: Building the CPI Data Cube

### What is a CPI (Coherent Processing Interval)?

A **CPI** is a sequence of $M$ chirps captured with:
- Fixed chirp parameters (BW, duration, center frequency)
- Consistent timing (same PRF)
- Phase coherence maintained across all chirps

### Data Structure

We'll build a 3D array (data cube):
- **Dimension 0**: Chirp index (slow-time) - size $M$
- **Dimension 1**: Sample index (fast-time) - size $N$
- **Dimension 2**: Channel (RX channel) - size 2 for CN0566

Shape: `(M, N, 2)`

#### Selecting `M` and `N`

When configuring a Range-Doppler radar experiment, two important parameters must be chosen:

- `N`: the number of fast-time samples collected during each chirp
- `M`: the number of chirps captured within one coherent processing interval (CPI)

Together, these parameters define the size of the Range-Doppler data cube and affect range coverage, velocity resolution, processing time, and memory usage.

##### Choosing `N`: Fast-Time Samples

`N` determines how many samples are collected during a single chirp. These samples form the input to the Range FFT, so `N` primarily affects the range axis.

For an indoor demonstration, targets are usually only a few metres from the radar. Since very long measurement ranges are not required, there is little benefit in collecting an excessive number of samples. Instead, choose an `N` that comfortably covers the desired range while keeping acquisition and processing fast.

A larger `N`:

- increases the number of FFT bins in the range dimension
- improves the visual smoothness of the range profile
- increases acquisition and processing time

Typical values are:

| `N` | Use case |
| --- | -------- |
| 512 | Suitable for short-range indoor demonstrations |
| 1024 | Good compromise between range detail and processing cost |
| 2048+ | Usually unnecessary for indoor experiments |

For this tutorial, we use:

```python
N = 1024
```

The PlutoSDR has min sampling frequency of aprox 600ksps. Lets assumen for now we will use an FFT size of 1024 that gives us a frequency resolution of:

$ \large
 \Delta f = \frac {F_s}{N} = \frac {600e3}{1024} = 585.94 Hz 
$  

----

##### Choosing M: Number of Chirps  

M determines how many chirps are captured during one CPI. These chirps form the input to the Doppler FFT, so M primarily affects the velocity axis.  
Velocity is measured by observing phase change from chirp to chirp. Capturing more chirps allows smaller Doppler frequency shifts to be resolved, improving velocity resolution.

A larger M:
- improves velocity resolution
- produces a smoother Doppler spectrum
- increases CPI duration
- makes the display less responsive to rapidly changing scenes

For this indoor demonstration, target velocities are expected to be relatively low, typically hand motion or walking-speed movement. Good Doppler resolution is therefore more useful than supporting very rapid scene updates.
Typical values are:
| M (Number of Chirps) | Use Case |
|----------------------|----------|
| 32 | Fast updates, coarse velocity resolution |
| 64 | Good compromise for indoor demonstrations |
| 128 | Improved Doppler resolution |
| 256 | Very good Doppler resolution, but slower updates |tes

For this tutorial, we use:
```python
M = 64
```

This provides enough Doppler resolution to distinguish stationary objects from slow-moving targets while keeping the display responsive.  

Practical Trade-Off  

The Range-Doppler map is generated from an M x N matrix:
`range_doppler_data.shape = (M, N)`  

Increasing either parameter improves part of the measurement, but also increases computation and memory requirements.  

For an indoor radar demonstration with ranges below approximately 10 m and human-scale motion, a practical starting point is:  

```python
N = 1024  # Samples per chirp
M = 64    # Chirps per CPI   
```   

These values produce a responsive Range-Doppler display while providing enough range and velocity resolution to visualize stationary objects, hand motion, and walking-speed targets.  

Engineering rationale:  
N primarily controls the range axis because it defines how much fast-time data is available for the Range FFT.  
 
M primarily controls the velocity axis because it defines how many chirps are available for the Doppler FFT.

We will want to capture both fast and slow time data in a single buffer.
Thus we need to allocated enough memory

Using the numbers we have select so far:  

$
M \times N =  64 \times 1024 = 65536 \text{ samples}
$

We will need twice this as we have two ADC channels.

What about sample rate. Well lets remind ourselves:

$
\Large f_b = k \frac{2R}{c}
$

where:
- R = range
- k = chirp slope
- c = speed of light

Lets use the slope from the prevous excercise `1.0 THz/s` and assume range will be no more than 10m

$$ \begin{aligned} 
f_b &= 1 \times 10^{12}\,\frac{2 \times 10}{3 \times 10^8} \\ 
    &= 66.7~\text{kHz} 
    \end{aligned} $$

We need sampling frequency 2x 66.7kHz.

Let set fs to 600ksps. This is the lowest frequency that the PlutoSDR supports.

Buffer Size        = 65536 samples  
Chirps per CPI     = 64  
Samples per Chirp  = 1024  

Sample Rate        = 600 kSPS  
Chirp Duration     = 1.707 ms  
Reset Time         = 100 µs  

PRI                = 1.807 ms  
PRF                = 553 Hz  

CPI Duration       = 115.6 ms
Velocity Resolution ≈ 0.13 m/s
`

#### Setup and Imports

### Hardware Configuration

Configure hardware, this is very similar to the previous notebook.
The main differences are:

- TDD is configured for 64 rising edges, each one separated by 1800uS
- ADF4159 ramp time is longer, that previous example.
- Tx/Rx Buffer is larger 65k

### CPI Capture Function

TODO: Implement CPI data capture

## Section 4: 2D FFT Implementation

### Step-by-Step 2D FFT Processing

1. **Range FFT** (1st dimension):
   - Apply window along fast-time (per chirp)
   - $N$-point FFT for each chirp
   - Result: Range profiles for each chirp

2. **Doppler FFT** (2nd dimension):
   - Apply window along slow-time (per range bin)
   - $M$-point FFT for each range bin
   - Result: Range-Doppler Map

### Windowing in Both Dimensions

- **Range dimension**: Reduces range sidelobes (e.g., Blackman window)
- **Doppler dimension**: Reduces velocity sidelobes (e.g., Hann window)

Trade-off: Resolution vs sidelobe suppression

TODO: Implement 2D FFT processing

## Section 5: Range-Doppler Map (RDM) Visualization

### Understanding the RDM

The Range-Doppler Map is a 2D image where:
- **X-axis**: Velocity (or Doppler frequency)
- **Y-axis**: Range
- **Color/Intensity**: Signal power (dB)

Each **bright spot** represents a target at a specific range and velocity.

### RDM Features to Look For

- **Zero Doppler column**: Static clutter (ground, walls, stationary objects)
- **Off-zero Doppler peaks**: Moving targets
- **Positive Doppler**: Targets approaching (coming toward RADAR)
- **Negative Doppler**: Targets receding (moving away)

TODO: Create RDM visualization function

### Single CPI Processing and Display

TODO: Capture, process, and display one RDM

In [ ]:
# TODO: Single RDM demo
# - Capture CPI
# - Process 2D FFT
# - Display RDM
# - Interpret results

## Section 6: Target Detection in Range-Doppler Space

### 2D Peak Detection

To extract target parameters:
1. Apply threshold to RDM
2. Find local maxima (peaks)
3. For each peak:
   - Range bin → range
   - Doppler bin → velocity
   - Peak magnitude → RCS/SNR

TODO: Implement 2D peak detection

## Section 7: MTI (Moving Target Indication) Filtering

### The Static Clutter Problem

In many scenarios, **static clutter** (walls, ground, stationary objects) dominates the RDM at **zero Doppler**, masking weak moving targets.

### MTI: High-Pass Filter in Slow-Time

MTI removes the zero-Doppler component by:
1. Subtracting mean across chirps (simple MTI)
2. Or applying high-pass filter along slow-time

$$
\text{MTI: } x_{\text{MTI}}[m, n] = x[m, n] - \frac{1}{M}\sum_{m=0}^{M-1} x[m, n]
$$

This **removes DC** in the slow-time domain → removes static clutter.

### Higher-Order MTI

For better clutter rejection:
- **2-pulse canceller**: $y[m] = x[m] - x[m-1]$
- **3-pulse canceller**: $y[m] = x[m] - 2x[m-1] + x[m-2]$

TODO: Implement MTI filtering

In [ ]:
# TODO: MTI filter
def apply_mti(data_cube, order=1):
    """
    Apply Moving Target Indication filter
    
    Parameters:
    -----------
    data_cube : ndarray
        Shape (M, N, channels)
    order : int
        MTI filter order (1=simple mean removal, 2=2-pulse canceller)
    
    Returns:
    --------
    data_mti : ndarray
        Clutter-suppressed data
    """
    # TODO: Implement
    # - Subtract mean across chirps (axis 0)
    # - Or apply differencing filter
    pass

### Comparison: With and Without MTI

TODO: Show side-by-side RDMs with/without MTI

In [ ]:
# TODO: MTI comparison
# - Process RDM without MTI
# - Process RDM with MTI
# - Plot side-by-side
# - Highlight moving target detection improvement

## Section 8: Range-Velocity Coupling

### The Coupling Problem

In triangular FMCW, there's a **coupling** between measured range and velocity:

- **Up-chirp**: $f_{\text{beat}} = f_R + f_D$
- **Down-chirp**: $f_{\text{beat}} = f_R - f_D$

Where:
- $f_R$ = range-induced beat frequency
- $f_D$ = Doppler shift

If we only use **up-chirps** (or only down-chirps), we can't separate $f_R$ and $f_D$.

### Solution: Up-Down Chirp Pairs

By processing both up and down chirps:

$$
f_R = \frac{f_{\text{up}} + f_{\text{down}}}{2}
$$

$$
f_D = \frac{f_{\text{up}} - f_{\text{down}}}{2}
$$

This **decouples** range from velocity.

### Implementation Note

For the current TDD system with sawtooth chirps:
- Coupling is small for low velocities
- For high-precision applications, implement triangular chirps and decouple

TODO: Demonstrate coupling effect

In [ ]:
# TODO: Range-velocity coupling demo
# - Simulate moving target at known range/velocity
# - Show apparent range shift due to Doppler
# - Calculate coupling error

## Section 9: Live Range-Doppler Demonstration

### Interactive RDM Display

TODO: Create live updating Range-Doppler map

In [ ]:
# TODO: Live RDM demo
# - Continuous CPI capture
# - Real-time 2D FFT processing
# - Update RDM display
# - Overlay detected targets
# - Display target list (range, velocity, SNR)
# - User can move target and observe changes

### Experiment Ideas

Try these experiments:

1. **Static target**: Place target, observe at zero Doppler
2. **Walking target**: Have someone walk toward/away from RADAR
3. **Hand waving**: Observe micro-Doppler from hand motion
4. **Multiple targets**: Have multiple people moving at different velocities
5. **MTI effectiveness**: Compare static clutter rejection with/without MTI

## Section 10: Summary and Next Steps

### What We've Accomplished

In this notebook, we:
1. ✓ Understood fast-time vs slow-time dimensions
2. ✓ Captured coherent multi-chirp CPIs
3. ✓ Implemented the 2D FFT (Range-Doppler processing)
4. ✓ Generated Range-Doppler Maps
5. ✓ Detected and tracked moving targets
6. ✓ Applied MTI filtering for clutter rejection
7. ✓ Measured both range AND velocity simultaneously

### Key Insights

- **2D FFT** provides simultaneous range-velocity measurement
- **More chirps** → better velocity resolution (but longer CPI)
- **MTI filtering** is essential for moving target detection in cluttered environments
- **Range-velocity coupling** exists but can be mitigated

### Current Capabilities

We can now:
- Detect multiple targets
- Measure range and velocity
- Reject static clutter
- Track moving objects

### Next Notebook: Beamforming Integration

In the next notebook (`6_FMCW_Beamforming_Integration.ipynb`), we'll:
- Add **angle estimation** using the phased array
- Implement **3D tracking**: Range + Velocity + Angle
- Use beam steering to focus on specific targets
- Create **Range-Angle-Doppler cubes**
- Demonstrate full RADAR tracking capabilities

This brings together everything: beamforming + FMCW + Doppler processing!

## Appendix A: 2D FFT Mathematics

### Discrete 2D FFT

For data matrix $x[m, n]$ of size $M \times N$:

$$
X[k, l] = \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x[m, n] \cdot e^{-j2\pi\left(\frac{km}{M} + \frac{ln}{N}\right)}
$$

### Separable Transform

The 2D FFT can be computed as two sequential 1D FFTs:

1. $Y[m, l] = \text{FFT}_N\{x[m, n]\}$ (FFT along each row)
2. $X[k, l] = \text{FFT}_M\{Y[m, l]\}$ (FFT along each column)

### Computational Complexity

- 1D FFT: $O(N \log N)$
- 2D FFT: $O(MN(\log M + \log N))$

For typical values ($M=64$, $N=512$):
- Total operations: ~$3 \times 10^5$ complex multiplies
- Real-time processing easily achievable on modern CPUs

## Appendix B: Doppler Ambiguity Resolution

### PRF Selection Trade-offs

| PRF | Advantage | Disadvantage |
|-----|-----------|-------------|
| High PRF | Large $v_{\text{max}}$ | Small $R_{\text{max}}$ |
| Low PRF | Large $R_{\text{max}}$ | Small $v_{\text{max}}$ |
| Medium PRF | Balanced | Ambiguities in both |

### Multiple PRF Technique

Use two or more PRFs and resolve ambiguities using Chinese Remainder Theorem:
1. Capture CPI at PRF₁
2. Capture CPI at PRF₂
3. Resolve velocity ambiguity algebraically

Extended unambiguous velocity:
$$
v_{\text{max,extended}} = v_{\text{max,1}} \times v_{\text{max,2}} / \gcd(v_{\text{max,1}}, v_{\text{max,2}})
$$

In [ ]:
import matplotlib.pyplot as plt
%matplotlib widget

test_fig, test_ax = plt.subplots(1, 1, figsize=(8, 4))
test_ax.plot([1, 2, 3], [1, 4, 9])
test_ax.set_title("Test Plot")

# Try displaying in a widget
test_layout = widgets.VBox([
    widgets.HTML("<h4>Test</h4>"),
    test_fig.canvas
])
display(test_layout)